# Run your first federated job

Complete [setup](setup.ipynb) first. You will run a two-client, three-round FedAvg recipe and verify the saved model and final evaluation.

`FedAvgRecipe` supplies the standard server workflow and client execution setup. `SimEnv` runs the server and both sites locally. The same client script executes at each site, but the site name selects different local samples.

In [ ]:
from pathlib import Path
from IPython.display import Code, display

chapter = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/fl_job.py").is_file())
code = chapter / "code"
print("Run the terminal commands from:", code)

In [ ]:
display(Code(filename=str(code / "fl_job.py"), language="python"))

In the activated terminal, change to the `code` directory printed above and run:

```bash
python fl_job.py
```

Expect three training rounds, evaluation of the saved global model at both sites, and a `Results:` path. Exact accuracy can vary across PyTorch versions. Successful completion, two site results, and a valid model file are the first checks—not a particular accuracy threshold.

The default workspace is `/tmp/nvflare/simulation/chapter1-none`. The optional `NVFLARE_SIMULATOR_WORKSPACE_ROOT` environment variable changes its parent directory; if you set it, give the notebook kernel the same value. Use a new root to keep multiple experiments separate.

In [ ]:
import os
from pathlib import Path

simulation_root = Path(os.environ.get("NVFLARE_SIMULATOR_WORKSPACE_ROOT", "/tmp/nvflare/simulation"))
workspace = simulation_root / "chapter1-none"
print(workspace)

In [ ]:
import json

server_run = workspace / "server/simulate_job"
model_file = server_run / "app_server/FL_global_model.pt"
evaluation_file = server_run / "cross_site_val/cross_val_results.json"
assert model_file.is_file(), f"Run python fl_job.py first; missing {model_file}"
results = json.loads(evaluation_file.read_text())
assert {"site-1", "site-2"} <= set(results)
final_scores = {site: scores["SRV_FL_global_model.pt"] for site, scores in results.items()}
print("Final model:", model_file)
print(json.dumps(final_scores, indent=2))

## Interpret the result

During training, each client reports `accuracy` for the **received global model before local training**. The server aggregates that metric alongside the returned locally trained weights. That metric therefore does not evaluate the newly aggregated weights from the same round.

`add_final_global_evaluation(recipe)` explicitly adds evaluation after training. The `SRV_FL_global_model.pt` entry above measures the saved final aggregate on each site's evaluation data. Additional entries may describe the selected best model; do not confuse those with the final model.

The dataset is intentionally easy. Better accuracy on these generated images verifies the mechanics, not performance on CIFAR-10 or private data. The maintained [Hello PyTorch example](../../../../../hello-world/hello-pt/README.md) offers an optional CIFAR-10 path when you are ready for a larger dataset.

**Try it:** run `python fl_job.py --num_rounds 1` in a fresh workspace, then compare final evaluation with three rounds. Explain why the final score cannot be inferred solely from the training-round metric.

Next: [convert local training to federated training](../01.2_convert_deep_learning_to_federated_learning/convert_dl_to_fl.ipynb).